# 04. 사진 바꾸기 — 배경 지우기 · 화질 4배 · 깊이

세 모델 모두 **사진 → 사진** 이다. 전처리(정규화)와 후처리(0~1 을 다시 사진으로)가 모델마다 다르다.

| 모델 | 입력 | 전처리 | 출력 |
|---|---|---|---|
| U2Net (onnx) | 320 x 320 RGB | ÷255 → (x − 평균) ÷ 표준편차 | 가림막(mask) 1채널 |
| SR-1032 | 작은 사진 480 x 270 + 크게 늘린 사진 1920 x 1080 | 없음 (0~255) | 1920 x 1080, 값 0~1 |
| Depth Anything V2 Small | 518 x 518 RGB | U2Net 과 같은 정규화 | 상대 깊이 (클수록 가깝다) |

## 1. 준비

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("GPU", "CPU")                        # 큰 사진을 다루는 모델 — GPU
G = MODELS / "gan"
MEAN = np.array([0.485, 0.456, 0.406], np.float32)  # ImageNet 평균 · 표준편차 (RGB)
STD = np.array([0.229, 0.224, 0.225], np.float32)

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 배경 지우기 — U2Net (ONNX 파일을 OpenVINO 가 바로 연다)

In [ ]:
u2 = core.compile_model(core.read_model(G / "u2net.onnx"), DEVICE)

def normalized(bgr, size):
    rgb = cv2.cvtColor(cv2.resize(bgr, (size, size)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    return ((rgb - MEAN) / STD).transpose(2, 0, 1)[None].astype(np.float32)

h, w = frame.shape[:2]
out = u2(normalized(frame, 320))[u2.output(0)]           # 첫 출력이 가장 고운 가림막
mask = out.squeeze().astype(np.float32)
mask = (mask - mask.min()) / (mask.max() - mask.min() + 1e-8)
mask = cv2.resize(mask, (w, h))[..., None]
cut = (frame * mask + 255 * (1 - mask)).astype(np.uint8)   # 배경을 흰색으로
show(np.hstack([frame, cv2.cvtColor((mask * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR), cut]))

## 4. 화질 4배 — 입력이 **두 개**인 모델
작은 사진(480 x 270)과, 그것을 그냥 크게 늘린 사진(1920 x 1080)을 함께 넣으면 "늘린 사진에 더할 세부"를 만들어 준다.
입력 크기가 고정이라 비율이 다른 사진은 letterbox(02 번)처럼 빈 곳을 채운다.

In [ ]:
sr = core.compile_model(G / "single-image-super-resolution-1032.xml", DEVICE)
print("입력:", [(i.any_name, tuple(i.shape)) for i in sr.inputs])

def super_res(bgr):
    h, w = bgr.shape[:2]
    s = min(480 / w, 270 / h)
    nw, nh = max(1, int(w * s)), max(1, int(h * s))
    small = cv2.copyMakeBorder(cv2.resize(bgr, (nw, nh)), 0, 270 - nh, 0, 480 - nw, cv2.BORDER_REPLICATE)
    big = cv2.resize(small, (1920, 1080), interpolation=cv2.INTER_CUBIC)
    feed = {sr.inputs[0].any_name: small.transpose(2, 0, 1)[None].astype(np.float32),
            sr.inputs[1].any_name: big.transpose(2, 0, 1)[None].astype(np.float32)}
    out = sr(feed)[sr.output(0)].squeeze().transpose(1, 2, 0)
    out = (out * 255).clip(0, 255).astype(np.uint8)
    return out[: nh * 4, : nw * 4], small[:nh, :nw]

big4, small = super_res(frame)
plain = cv2.resize(small, (big4.shape[1], big4.shape[0]), interpolation=cv2.INTER_CUBIC)
cy, cx = big4.shape[0] // 2, big4.shape[1] // 2          # 가운데 부분을 크게 비교
crop = lambda im: im[cy - 120: cy + 120, cx - 160: cx + 160]
print("왼쪽: 그냥 늘림(bicubic) | 오른쪽: 모델")
show(np.hstack([crop(plain), crop(big4)]))

## 5. 깊이 — 한 장의 사진에서 멀고 가까움

In [ ]:
depth = core.compile_model(G / "depth-v2s.xml", DEVICE)
d = depth(normalized(frame, 518))[depth.output(0)].squeeze().astype(np.float32)   # 518 = 14 × 37
d = cv2.resize(d, (w, h))
near = (d - d.min()) / (d.max() - d.min() + 1e-8)             # 이 사진 안에서만 0~1 (상대 깊이)
color = cv2.applyColorMap((near * 255).astype(np.uint8), cv2.COLORMAP_INFERNO)
show(np.hstack([frame, color]))

## 해 볼 것
1. 배경 지우기에서 가림막을 0.5 기준으로 딱 자르면(`mask > 0.5`) 경계가 어떻게 달라지나?
2. 배경을 흰색 대신 다른 사진으로 바꿔 보자 (`frame2 = grab()` 후 크기를 맞춰 합성).
3. 깊이에서 "가장 가까운 20%" 만 남기고 나머지를 흐리게(`cv2.GaussianBlur`) 해서 인물 사진 모드를 만들어 보자.
4. 세 모델을 `"CPU"` 와 `"GPU"` 에서 각각 재 보자.